$$
\begin{array}{c}
\textbf{Linear Regression - Spring 2026}\\\\
\textit{Center for Data Science, New York University} \\\\
\textit{March 06, 2026}\\\\\
\text{ Prepared by: Vivek Kumar Agarwal}\\\\
\textbf{Recitation 6: Linear Regression}
\end{array}
$$

---

![Linear Regression](../figures/LR_lab6_image1.png)

---

# Midterm Revision — Linear Regression Models
### Spring 2026

---

## Outline

1. **PDF vs PMF** — Continuous vs Discrete Random Variables
2. **BLP Derivation & FOCs** — Solving for β₀ and β₁
3. **CEF & BLP/BLA Connection** — When do they coincide?
4. **OLS Consistency & WLLN** — Why OLS works with iid data
5. **R²** — Definition, bounds, and behavior when adding regressors
6. **Homo/Heteroskedasticity** — What it means and why it matters
7. **Standard Errors** — Default, robust, and clustered
8. **Omitted Variable Bias** — Intuition and algebra
9. **Train / Validation / Test Splits** — Model selection for prediction
10. **Multiple Linear Regression** — Interpreting MLR coefficients and FWL Theorem

---


## 1. PDF vs PMF

### Discrete RV → PMF
$$f(x) = P(X = x)$$
$$\sum_x f(x) = 1$$

### Continuous RV → PDF
$$P(X = x) = 0 \quad \text{for all } x$$

Probability only comes from integrating over an interval:
$$P(a \leq X \leq b) = \int_a^b f(x)\, dx$$
$$\int_{-\infty}^{\infty} f(x)\, dx = 1$$

The PDF $f(x)$ describes **density**, not probability — just like how water density 
at a single point tells you nothing about volume. It is therefore never the case 
that $f(x) = P(X = x)$ for a continuous RV, even if $f(x)$ happens to be 
a "nice" number like 0.5.

---

## 2. BLP Derivation & First Order Conditions

### The Setup
We want to find the **Best Linear Predictor** of Y given X:
$$Y = \beta_0 + \beta_1 X + \varepsilon$$

"Best" means minimizing the **Mean Squared Error**:
$$\min_{\beta_0, \beta_1} E[(Y - \beta_0 - \beta_1 X)^2]$$

### First Order Conditions (FOCs)
Taking derivatives and setting to zero gives us two conditions:

$$E[Y - \beta_0 - \beta_1 X] = 0 \tag{1}$$
$$E[X(Y - \beta_0 - \beta_1 X)] = 0 \tag{2}$$

### Solving the System
From (1):
$$E[Y] = \beta_0 + \beta_1 E[X]$$
$$\beta_0 = E[Y] - \beta_1 E[X]$$

Plugging into (2) and expanding:
$$E[XY] - \beta_0 E[X] - \beta_1 E[X^2] = 0$$
$$E[XY] - (E[Y] - \beta_1 E[X])E[X] - \beta_1 E[X^2] = 0$$
$$E[XY] - E[X]E[Y] = \beta_1(E[X^2] - E[X]^2)$$

Recognizing the covariance and variance:
$$\boxed{\beta_1 = \frac{Cov(X,Y)}{Var(X)}}$$
$$\boxed{\beta_0 = E[Y] - \beta_1 E[X]}$$



Note: When plugging in a specific DGP (e.g. $Y = 3X$), substitute directly into 
the FOCs and solve — the algebra will always collapse cleanly.

---

## 3. CEF & BLP/BLA Connection

### The CEF
The **Conditional Expectation Function (CEF)** is:
$$E[Y|X]$$
It gives the average value of Y for each value of X. It is the 
**best predictor** of Y given X — but it can be nonlinear.

### The BLP & BLA
The **Best Linear Predictor (BLP)** is the best predictor of Y 
restricted to be linear in X:
$$\beta_1 = \frac{Cov(X,Y)}{Var(X)}, \quad \beta_0 = E[Y] - \beta_1 E[X]$$

The **Best Linear Approximation (BLA)** is the linear function 
that best approximates the CEF $E[Y|X]$:
$$\min_{\beta_0, \beta_1} E[(E[Y|X] - \beta_0 - \beta_1 X)^2]$$

### The Key Results

$$\boxed{BLP = BLA \text{ always}}$$

Both optimization problems always yield the **same** $\beta_0$ and $\beta_1$. 

$$\boxed{BLP = CEF \text{ only when } E[Y|X] \text{ is linear in X}}$$

When the CEF is nonlinear, the BLP/BLA is simply the best linear 
approximation to it — not the CEF itself.

### The Law of Iterated Expectations
$$E[Y] = E[E[Y|X]]$$
Unconditional expectations can always be recovered by averaging the CEF over X.

---

## 4. OLS Consistency & WLLN

### The Setup
In practice we don't observe the true parameters $\beta_0, \beta_1$ — 
we **estimate** them from a sample of n observations using OLS:
$$\hat{\beta}_1 = \frac{\sum_i (X_i - \bar{X})(Y_i - \bar{Y})}{\sum_i (X_i - \bar{X})^2}$$

### The WLLN
For an **iid** sample, the **Weak Law of Large Numbers** tells us:
$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{p} E[X] \quad \text{as } n \to \infty$$

Sample averages converge in probability to their population counterparts 
as the sample size grows.

### OLS Consistency
Because $\hat{\beta}_1$ is a function of sample averages, the WLLN + CMT 
(Continuous Mapping Theorem) together give us:

$$\hat{\beta}_1 \xrightarrow{p} \frac{Cov(X,Y)}{Var(X)} = \beta_1$$

This requires only that the sample is **iid** — no assumptions about 
normality or homoskedasticity needed for consistency.

### Descriptive vs Causal Consistency
- **Descriptive/Predictive:** iid sample is sufficient for consistency
- **Causal:** additionally need $E[XU] = 0$ — if violated, OLS is 
inconsistent even with infinite data (OVB)

> The WLLN is why large samples help — but only if the data is iid. 
> A biased sample stays biased no matter how large it gets.

---

## 5. R²— Definition, Bounds & Behavior

### The Decomposition
The total variation in Y can be decomposed as:
$$\underbrace{TSS}_{Total} = \underbrace{ESS}_{Explained} + \underbrace{SSR}_{Residual}$$

$$\sum_i(Y_i - \bar{Y})^2 = \sum_i(\hat{Y}_i - \bar{Y})^2 + \sum_i\hat{\varepsilon}_i^2$$

### R² Definition
$$\boxed{R^2 = \frac{ESS}{TSS} = 1 - \frac{SSR}{TSS}}$$

The share of variation in Y **explained** by the model.

### Bounds
$$0 \leq R^2 \leq 1$$

- $R^2 = 1$ — perfect fit, all variation explained
- $R^2 = 0$ — model explains nothing, no better than predicting $\bar{Y}$
- $R^2$ can **never** be negative or exceed 1

### Behavior When Adding Regressors
$$R^2 \text{ weakly increases when adding regressors — always}$$

Adding a new variable can never hurt R² in-sample because OLS 
can always set its coefficient to zero if it adds no explanatory power. 
This is why R² is a **poor measure of out-of-sample performance** — 
a model with more regressors will always look better in-sample 
even if it is overfitting.

### Out-of-Sample Performance
For predictive models, **MSE on a test set** is preferred over R²:
$$MSE = \frac{1}{n}\sum_i(Y_i - \hat{Y}_i)^2$$



---

## 6. Homo/Heteroskedasticity

### Definition
Homoskedasticity means the variance of the error term is **constant** across all values of X:
$$Var(\varepsilon | X) = \sigma^2 \quad \text{for all } X$$

Heteroskedasticity means the variance of the error term **changes** with X:
$$Var(\varepsilon | X) = \sigma^2(X) \quad \text{varies with } X$$

### Why it Matters
OLS estimates of $\beta_0$ and $\beta_1$ remain **consistent** under both — 
heteroskedasticity does not bias our estimates. What changes is the 
**standard errors**:

| | Homoskedastic | Heteroskedastic |
|---|---|---|
| **OLS estimates** | Consistent | Consistent |
| **Default SEs** | Correct | Incorrect |
| **Robust SEs** | Conservative | Correct |
| **BLUE** | OLS is BLUE | OLS is not BLUE |

### Gauss-Markov & BLUE
Under homoskedasticity + iid + linear CEF, OLS is the 
**Best Linear Unbiased Estimator (BLUE)** — minimum variance 
among all linear unbiased estimators. Heteroskedasticity breaks this.

### Point to remember
The key is to ask: *does the spread of Y around the regression line 
change as X changes?*

- **Shoe size ~ Height** — spread is roughly constant → homoskedastic
- **Income ~ Age** — young people cluster near low income, older people 
spread widely → heteroskedastic
- **Baseball card value ~ Year** — older cards vary wildly in value, 
newer cards are more uniform → heteroskedastic

---

## 7. Standard Errors — Default, Robust & Clustered

### Why Standard Errors Matter
OLS gives us $\hat{\beta}_1$ — but how uncertain are we about it?
Standard errors measure the **spread of the sampling distribution** of $\hat{\beta}_1$.
Getting them wrong means wrong confidence intervals and wrong inference.

### Default Standard Errors
Assume homoskedasticity — $Var(\varepsilon|X) = \sigma^2$ constant across X.
Only correct when this assumption holds. If violated, default SEs are 
**inconsistent** — they can be too small or too large.

### Heteroskedasticity-Robust Standard Errors
Make no assumption about the variance of $\varepsilon$ across X.
Always safer than default — if homoskedasticity holds, robust SEs are 
slightly conservative but still valid.

$$\text{When in doubt} \rightarrow \text{use robust SEs}$$

### Clustered Standard Errors
Used when data has a **group structure** where observations within 
the same group are correlated:
$$Cov(\varepsilon_i, \varepsilon_j) \neq 0 \quad \text{if } i,j \text{ in same cluster}$$

Examples of natural clusters:
- Students within the same school
- Players within the same team
- Siblings within the same family

Clustered SEs account for within-group correlation. 
Failing to cluster when groups exist **understates** true uncertainty.

### The Decision Rule
| Situation | SE Choice |
|---|---|
| Constant variance, no groups | Default |
| Variance changes with X | Robust |
| Group structure in data | Clustered |

> Clustered SEs require **enough clusters** to be reliable — 
> with very few clusters (e.g. 5), they break down.

---

## 8. Omitted Variable Bias (OVB)

### The Setup
Suppose the **true causal model** is:
$$Y = \alpha_0 + \alpha_1 X_1 + \alpha_2 X_2 + U$$
where $E[U] = E[X_1 U] = E[X_2 U] = 0$

But we **omit** $X_2$ and estimate instead:
$$Y = \alpha_0^* + \alpha_1^* X_1 + U^*$$

### The OVB Formula
OLS will consistently estimate:
$$\boxed{\alpha_1^* = \alpha_1 + \alpha_2 \frac{Cov(X_1, X_2)}{Var(X_1)}}$$

The bias term is $\alpha_2 \frac{Cov(X_1, X_2)}{Var(X_1)}$ — it disappears only if 
$\alpha_2 = 0$ (X_2 doesn't belong) or $Cov(X_1, X_2) = 0$ (X_2 is uncorrelated with X_1).

### Sign of the Bias
The sign of OVB depends on two things:

| $\alpha_2$ | $Cov(X_1, X_2)$ | Bias |
|---|---|---|
| Positive | Positive | Upward |
| Positive | Negative | Downward |
| Negative | Positive | Downward |
| Negative | Negative | Upward |

### Deriving OVB Algebraically
Starting from $\frac{Cov(X_1, Y)}{Var(X_1)}$ and substituting the true DGP:

$$\frac{Cov(X_1, Y)}{Var(X_1)} = \frac{Cov(X_1, \alpha_0 + \alpha_1 X_1 + \alpha_2 X_2 + U)}{Var(X_1)}$$

$$= \frac{\alpha_1 Cov(X_1, X_1) + \alpha_2 Cov(X_1, X_2) + Cov(X_1, U)}{Var(X_1)}$$

$$= \frac{\alpha_1 Var(X_1) + \alpha_2 Cov(X_1, X_2) + 0}{Var(X_1)}$$

$$= \alpha_1 + \alpha_2\frac{Cov(X_1, X_2)}{Var(X_1)}$$

### Intuition
OVB occurs because the omitted variable $X_2$ ends up "hiding" 
inside $U^*$. OLS then picks up not just the effect of $X_1$ on Y, 
but also the indirect effect running through the correlation 
between $X_1$ and $X_2$.

**Example:** Regressing Income on Education, omitting Ability.
Ability drives both Education and Income — so $\hat{\alpha}_1$ 
will be upward biased, capturing some of Ability's effect.

---

## 9. Train / Validation / Test Splits

### The Problem with In-Sample Fit
R² always increases with more regressors — so we can't use 
in-sample fit to choose between models. We need **out-of-sample** 
evaluation.

### The Three-Way Split
$$\underbrace{\text{All Data}}_{} \rightarrow \begin{cases} \text{Training Set} \\ \text{Validation Set} \\ \text{Test Set} \end{cases}$$

### What Each Set Does

**Training Set**
- Fit the model — estimate $\hat{\beta}_0, \hat{\beta}_1, ...$
- Done separately for each candidate model

**Validation Set**
- Compare candidate models **after** training
- Pick the model with the lowest validation MSE
- e.g. choosing between polynomial orders 1 through 5

**Test Set**
- Used **once** at the very end
- Gives an honest estimate of true out-of-sample performance
- Never used for model selection — otherwise it leaks into the process

### Why Keep Test Set Separate?
If you use the test set to select your model, it effectively 
becomes another validation set — your final MSE estimate is 
then optimistic and no longer honest.

$$\text{Validation} \rightarrow \text{model selection}$$
$$\text{Test} \rightarrow \text{final honest evaluation, used once}$$

---

## 10. Multiple Linear Regression (MLR)

### The Setup
Extending SLR to multiple regressors:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_k X_k + \varepsilon$$

The FOCs generalize naturally — for each regressor $X_j$:
$$E[Y - \beta_0 - \beta_1 X_1 - ... - \beta_k X_k] = 0$$
$$E[X_j(Y - \beta_0 - \beta_1 X_1 - ... - \beta_k X_k)] = 0$$

### Interpreting MLR Coefficients
$$\beta_j = \text{effect of } X_j \text{ on Y, holding all other regressors constant}$$

This is the key difference from SLR — each coefficient is a 
**partial effect**, not a total effect.

### The Frisch-Waugh-Lovell (FWL) Theorem
The MLR coefficient $\hat{\beta}_1$ can be recovered in three steps:

1. Regress $X_1$ on all other regressors → get residuals $\tilde{X}_1$
2. Regress $Y$ on all other regressors → get residuals $\tilde{Y}$
3. Regress $\tilde{Y}$ on $\tilde{X}_1$ → coefficient = $\hat{\beta}_1$

$$\boxed{\hat{\beta}_1^{MLR} = \frac{Cov(\tilde{X}_1, \tilde{Y})}{Var(\tilde{X}_1)}}$$

$\tilde{X}_1$ is $X_1$ **purged** of all variation explained by other regressors.
FWL tells us MLR is really just SLR on residualized variables.

### Connection to OVB
When we go from MLR → SLR by dropping $X_2$, the SLR coefficient 
picks up the effect of $X_2$ running through its correlation with $X_1$ 
— that's exactly the OVB term:
$$\hat{\beta}_1^{SLR} = \hat{\beta}_1^{MLR} + \hat{\beta}_2 \frac{Cov(X_1, X_2)}{Var(X_1)}$$

---

## Summary — Midterm Revision

| Topic | Key Takeaway |
|---|---|
| **PDF vs PMF** | For continuous RVs, $f(x) \neq P(X=x)$. PDF is density, not probability |
| **BLP & FOCs** | Two FOCs → $\beta_1 = \frac{Cov(X,Y)}{Var(X)}$, $\beta_0 = E[Y] - \beta_1 E[X]$ |
| **CEF & BLP/BLA** | BLP = BLA always. BLP = CEF only when CEF is linear in X |
| **OLS Consistency** | iid + WLLN + CMT → $\hat{\beta}_1 \xrightarrow{p} \beta_1$. Causal consistency additionally needs $E[XU]=0$ |
| **R²** | $R^2 = 1 - \frac{SSR}{TSS}$, weakly increases with more regressors — poor out-of-sample measure |
| **Homo/Heteroskedasticity** | OLS stays consistent either way — only standard errors are affected |
| **Standard Errors** | Default → homoskedastic. Robust → heteroskedastic. Clustered → group structure |
| **OVB** | $\alpha_1^* = \alpha_1 + \alpha_2\frac{Cov(X_1,X_2)}{Var(X_1)}$ — bias disappears only if $\alpha_2=0$ or $Cov(X_1,X_2)=0$ |
| **Train/Val/Test** | Train → fit. Validation → model selection. Test → honest final evaluation, used once |
| **MLR & FWL** | Each $\beta_j$ is a partial effect. FWL recovers $\hat{\beta}_1^{MLR}$ by residualizing both $X_1$ and $Y$ on other regressors |

---

## Office Hours Friday 11 AM - 1 PM  Room No - 244
or dropin your questions at vka244@nyu.edu

---